In [95]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [97]:
df = pd.read_csv('lendingclub.csv')
df.head()

,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,...,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag
0,73582,3500.0,3500.0,225.0,36 months,10.28%,113.39,C,C1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
1,87023,7500.0,7500.0,800.0,36 months,13.75%,255.43,E,E2,Evergreen Center,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
2,70686,5000.0,5000.0,0.0,36 months,7.75%,156.11,A,A3,Homemaker,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
3,83489,2600.0,2600.0,575.0,36 months,8.38%,81.94,A,A5,College Pro Painters,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
4,74323,6500.0,6500.0,0.0,36 months,9.64%,208.66,B,B4,Air Force,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N


In [101]:
# 부도를 어떤 값으로 살펴볼 것인가?
# Charged off/Default: 부도(=1), Fully Paid: 부도 아님(=0)

df['default'] = df['loan_status'].apply(lambda x: 1 if x in ['Charged Off','Default'] else 0)

# 정보집합 관점에서의 사용 가능한 변수의 선택
features = ['acc_now_delinq', 'acc_open_past_24mths', 'addr_state', 'all_util', 'annual_inc', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths',
    'collection_recovery_fee', 'collections_12_mths_ex_med', 'delinq_2yrs', 'delinq_amnt',
    'dti', 'emp_length', 'emp_title', 'fico_range_high', 'fico_range_low',
    'home_ownership', 'il_util', 'inq_fi', 'inq_last_12m', 'inq_last_6mths', 'installment',
    'last_fico_range_high', 'last_fico_range_low', 'max_bal_bc', 'mo_sin_old_il_acct',
    'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc',
    'mths_since_last_delinq', 'mths_since_last_major_derog', 'mths_since_last_record',
    'mths_since_rcnt_il', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd',
    'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl',
    'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
    'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'open_acc', 'open_acc_6m',
    'open_il_12m', 'open_il_24m', 'open_act_il', 'open_rv_12m', 'open_rv_24m', 'pct_tl_nvr_dlq',
    'percent_bc_gt_75', 'pub_rec', 'pub_rec_bankruptcies', 'purpose', 'revol_bal', 'revol_util', 'tax_liens',
    'term', 'tot_coll_amt', 'tot_cur_bal', 'tot_hi_cred_lim', 'total_acc', 'total_bal_ex_mort',
    'total_bal_il', 'total_bc_limit', 'total_cu_tl', 'total_il_high_credit_limit',
    'total_rev_hi_lim', 'verification_status'
]

# 최종 분석에 사용할 변수 추린 후, 약간의 전처리
df= df[features + ['default','int_rate','term','verification_status']].dropna()
df['loan_duration_years']=df['term'].apply(lambda x: 3 if '36' in x else 5)

In [109]:
# 4. 의미적 수치형 문자 처리
if 'term' in df.columns:
    df['loan_duration_years'] = df['term'].apply(lambda x: 3 if '36' in x else 5)

if 'int_rate' in df.columns:
    df['int_rate_num'] = df['int_rate'].str.rstrip('%').astype(float) / 100
if 'int_rate' in df.columns:
    df.drop(columns=['int_rate'], inplace=True)

if 'revol_util' in df.columns:
    df['revol_util_num'] = df['revol_util'].str.rstrip('%').astype(float) / 100
    df.drop(columns=['revol_util'], inplace=True)

# # 실제 존재하는 열만 필터링해서 삭제
# df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)

# 5. 최종 분류 재확인

numeric_cols_final = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
string_cols_final = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]

print("최종 숫자형 변수:")
print(numeric_cols_final)
print("\n최종 문자형 변수:")
print(string_cols_final)

최종 숫자형 변수:
['acc_now_delinq', 'acc_open_past_24mths', 'all_util', 'annual_inc', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'collection_recovery_fee', 'collections_12_mths_ex_med', 'delinq_2yrs', 'delinq_amnt', 'dti', 'fico_range_high', 'fico_range_low', 'il_util', 'inq_fi', 'inq_last_12m', 'inq_last_6mths', 'installment', 'last_fico_range_high', 'last_fico_range_low', 'max_bal_bc', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_last_delinq', 'mths_since_last_major_derog', 'mths_since_last_record', 'mths_since_rcnt_il', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'open_acc', 'open

In [111]:
for col in string_cols_final:
    print(f"===== {col} =====")
    print(df[col].value_counts(dropna=False).head(10))  # 상위 10개 + NaN 포함
    print(f"총 고유값 수: {df[col].nunique(dropna=True)}")
    print()

===== addr_state =====
addr_state
CA    1659
NY     971
FL     767
TX     690
NJ     462
PA     402
GA     386
IL     382
MA     381
VA     372
Name: count, dtype: int64
총 고유값 수: 50

===== emp_length =====
emp_length
< 1 year     1826
10+ years    1677
2 years      1336
1 year       1286
3 years       965
4 years       763
5 years       639
6 years       441
7 years       365
8 years       355
Name: count, dtype: int64
총 고유값 수: 11

===== emp_title =====
emp_title
NaN                      606
Self Employed             40
Self                      28
Retired                   23
US Air Force              22
US Army                   20
IBM                       18
Fidelity Investments      16
Self-employed             16
Bank of America Corp.     14
Name: count, dtype: int64
총 고유값 수: 8015

===== home_ownership =====
home_ownership
RENT        5107
MORTGAGE    3898
OWN          852
OTHER        135
NONE           8
Name: count, dtype: int64
총 고유값 수: 5

===== purpose =====
purpose
debt_con

In [113]:
# categorical data의 더미변수 처리
df = pd.get_dummies(df, columns = ['addr_state', 'emp_length', 'emp_title', 'home_ownership', 'purpose', 'verification_status'], drop_first=True)
df = pd.get_dummies(df, columns = ['term'], drop_first=True)

In [115]:
df['expected_return'] = (1 + df['int_rate_num']) ** df['loan_duration_years'] - 1

In [117]:
# Train-test-split:

X = df.drop(columns=['default'])
y = df['default']

X_temp, X_test, y_temp, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp, test_size=0.25, random_state=42)

original_temp, original_test = train_test_split(original_data, test_size=0.2, random_state=42)
original_train, original_val = train_test_split(original_temp, test_size=0.25, random_state=42)

In [119]:
# 변수의 표준화 (LASSO 사용 시, 변수의 scale을 맞춰주기 위함.)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
continuous_features = ['acc_now_delinq', 'acc_open_past_24mths', 'all_util', 'annual_inc', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths',
    'collection_recovery_fee', 'collections_12_mths_ex_med', 'delinq_2yrs', 'delinq_amnt',
    'dti', 'fico_range_high', 'fico_range_low',
    'il_util', 'inq_fi', 'inq_last_12m', 'inq_last_6mths', 'installment',
    'last_fico_range_high', 'last_fico_range_low', 'max_bal_bc', 'mo_sin_old_il_acct',
    'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc',
    'mths_since_last_delinq', 'mths_since_last_major_derog', 'mths_since_last_record',
    'mths_since_rcnt_il', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd',
    'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl',
    'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
    'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'open_acc', 'open_acc_6m',
    'open_il_12m', 'open_il_24m', 'open_act_il', 'open_rv_12m', 'open_rv_24m', 'pct_tl_nvr_dlq',
    'percent_bc_gt_75', 'pub_rec', 'pub_rec_bankruptcies', 'revol_bal', 'tax_liens',
    'tot_coll_amt', 'tot_cur_bal', 'tot_hi_cred_lim', 'total_acc', 'total_bal_ex_mort',
    'total_bal_il', 'total_bc_limit', 'total_cu_tl', 'total_il_high_credit_limit',
    'total_rev_hi_lim'
]

X_train[continuous_features]=scaler.fit_transform(X_train[continuous_features])
X_val[continuous_features]=scaler.transform(X_val[continuous_features])


In [125]:
## Train set의 balance
from sklearn.utils import resample
from sklearn.linear_model import Lasso
from sklearn.linear_model import LassoCV

X_train_balanced = pd.concat([X_train, y_train],axis=1)
default_data = X_train_balanced[X_train_balanced['default']==1]
non_default_data = X_train_balanced[X_train_balanced['default']==0]

non_default_downsampled =  resample(
    non_default_data,
    replace=False,
    n_samples = len(default_data),
    random_state=42)

balanced_train_data = pd.concat([default_data, non_default_downsampled])

X_train = balanced_train_data.drop(columns=['default'])
y_train = balanced_train_data['default']

In [127]:
# 1. 최적 alpha 찾기
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train_balanced, y_train)

best_alpha = lasso_cv.alpha_
print("✅ Best alpha:", best_alpha)

# 2. 그걸로 Lasso 모델 학습
lasso = Lasso(alpha=best_alpha, max_iter=10000)
lasso.fit(X_train_balanced, y_train)

ValueError: Input X contains NaN.
LassoCV does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [80]:
# 4. best alpha로 다시 학습
lasso = Lasso(alpha=best_alpha, max_iter=10000)
lasso.fit(X_train_balanced, y_train)

# 5. 예측
y_val_pred = lasso.predict(X_val_scaled)

# 6. Sharpe Ratio 최대화하는 threshold 찾기
thresholds = np.linspace(min(y_val_pred), max(y_val_pred), 100)
best_sharpe = -np.inf
best_threshold = None

for t in thresholds:
    mask = y_val_pred > t
    if mask.sum() > 1:
        returns = y_val[mask]
        if returns.std() > 0:  # division by zero 방지
            sharpe = returns.mean() / returns.std()
            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_threshold = t

print(f"Best alpha: {best_alpha}")
print(f"Best threshold: {best_threshold}")
print(f"Best Sharpe Ratio: {best_sharpe:.4f}")

NameError: name 'best_alpha' is not defined

In [ ]:
# 1. 테스트셋도 스케일링
X_test_scaled = scaler.transform(X_test)

# 2. 예측
y_test_pred = lasso.predict(X_test_scaled)

# 3. threshold 적용해 포트폴리오 구성
mask = y_test_pred > best_threshold
portfolio = y_test[mask]

# 4. 샤프비율 계산
if portfolio.std() > 0:
    test_sharpe = portfolio.mean() / portfolio.std()
else:
    test_sharpe = np.nan  # 또는 0

print(f"[Test] Portfolio size: {len(portfolio)}")
print(f"[Test] Sharpe Ratio: {test_sharpe:.4f}")